In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
import warnings
from sklearn.exceptions import ConvergenceWarning

In [2]:
# ==============================================================================
# 1. CONFIGURATION & DATA (The Control Panel)
# ==============================================================================
database = {
    "Function 1": {
        "beta": 5,       # Increased to 5 in week 4
        "nu": 2.5,          # Standard smoothness (2.5 = smooth, 1.5 = rough)
        "data": [
            [0.31940389, 0.76295937, 1.32267704e-079],
            [0.57432921, 0.8798981, 1.03307824e-046],
            [0.73102363, 0.73299988, 7.71087511e-016],
            [0.84035342, 0.26473161, 3.34177101e-124],
            [0.65011406, 0.68152635, -3.60606264e-003],
            [0.41043714, 0.1475543, -2.15924904e-054],
            [0.31269116, 0.07872278, -2.08909327e-091],
            [0.68341817, 0.86105746, 2.53500115e-040],
            [0.08250725, 0.40348751, 3.60677119e-081],
            [0.88388983, 0.58225397, 6.22985647e-048],
            # New weekly data points
            [0.284982, 0.224270,-2.4911192925749473e-42],
            [0.825462, 0.858263, 1.077784e-65],
            [0.929177, 0.174308, 8.913961e-207], 
            [0.389886, 0.143540, 1.583643e-56],
            [0.626620, 0.864894, -3.014412e-39], # Week 5
            [0.840600, 0.727544, 7.012427379270183e-39], # Week 6
            [0.654510, 0.996117, -2.403238e-95], # Week 7
            [0.142376, 0.267191, 3.636179e-72], # Week 8
            [0.887883, 0.583817, 1.885812e-49], # Week 9
            [0.248619, 0.396690, 2.891293e-22], # Week 10
            [0.411448, 0.326030, -2.875281e-07], # Week 11
            [0.149754, 0.102835, -1.793577e-123], # Week 12
        ]
    },
    
    "Function 2": {
        # Changed beta to 1.0 in week 4 to exploit peak 
        # beta to 1.5 week 5
        # beta 0.1 week 6
        # beta to 0.05 week 7
        #beta to 0.01 week 8
        #beta to 0.1 week 10
        #beta to 0.5 week 11
        #beta to 0.1 week 13
        "beta": 0.1, "nu": 2.5, 
        "data": [
            [ 0.66579958,  0.12396913,  0.53899612],
            [ 0.87779099,  0.7786275 ,  0.42058624],
            [ 0.14269907,  0.34900513, -0.06562362],
            [ 0.84527543,  0.71112027,  0.29399291],
            [ 0.45464714,  0.29045518,  0.21496451],
            [ 0.57771284,  0.77197318,  0.02310555],
            [ 0.43816606,  0.68501826,  0.24461934],
            [ 0.34174959,  0.02869772,  0.03874902],
            [ 0.33864816,  0.21386725, -0.01385762],
            [ 0.70263656,  0.9265642 ,  0.61120522],
            # New weekly data points
            [0.226591, 0.938833, 0.010831832870674957],
            [0.702895, 0.931300, 0.7069941],
            [0.478696, 0.512233, 0.7941273],
            [0.473497, 0.506634, 0.6200051],
            [0.066240, 0.518910, -0.0873039], # Week 5
            [0.480320, 0.509709, 0.7291344505925994], # Week 6
            [0.481616, 0.520967, 0.7501247], # Week 7
            [0.482767, 0.513828, 0.7215644], # Week 8
            [0.476954, 0.514491, 0.5215702], # Week 9
            [0.386535, 0.565820, 0.0905378], # Week 10
            [0.481210, 0.512741, 0.5321660], # Week 11
            [0.067130, 0.624215, -0.0954258], # Week 12
        ]
    },

    "Function 3": {
        # changed beta to 2.5 in week 4 as still stuck in negative values
        #beta to 5 week 5
        #beta 2.5 week 8
        #beta to 5 week 9
        #beta to 2.5 week 10
        #beta to 5 week 12
        "beta": 5, "nu": 2.5, 
        "data": [
            [ 0.17152521,  0.34391687,  0.2487372 , -0.1121222 ],
            [ 0.24211446,  0.64407427,  0.27243281, -0.08796286],
            [ 0.53490572,  0.39850092,  0.17338873, -0.11141465],
            [ 0.49258141,  0.61159319,  0.34017639, -0.03483531],
            [ 0.13462167,  0.21991724,  0.45820622, -0.04800758],
            [ 0.34552327,  0.94135983,  0.26936348, -0.11062091],
            [ 0.15183663,  0.43999062,  0.99088187, -0.39892551],
            [ 0.64550284,  0.39714294,  0.91977134, -0.11386851],
            [ 0.74691195,  0.28419631,  0.22629985, -0.13146061],
            [ 0.17047699,  0.6970324 ,  0.14916943, -0.09418956],
            [ 0.22054934,  0.29782524,  0.34355534, -0.04694741],
            [ 0.66601366,  0.67198515,  0.2462953 , -0.10596504],
            [ 0.04680895,  0.23136024,  0.77061759, -0.11804826],
            [ 0.60009728,  0.72513573,  0.06608864, -0.03637783],
            [ 0.96599485,  0.86111969,  0.56682913, -0.05675837],
            # New weekly data points
            [0.396138, 0.394802, 0.493251, -0.009530784464706916],
            [0.183060, 0.856976, 0.189214, -0.1533000],
            [0.388658, 0.138724, 0.486308, -0.0299720], # Week 3
            [0.841280, 0.995980, 0.001381, -0.1510810],
            [0.740753, 0.503364, 0.230483, -0.1101017], # Week 5
            [0.266801, 0.668468, 0.281191, -0.08105988182987513], # Week 6
            [0.389238, 0.133323, 0.485236, -0.0373759], # Week 7
            [0.324966, 0.263817, 0.565533, -0.0618252], # Week 8
            [0.564165, 0.274821, 0.494539, -0.0339189], # Week 9
            [0.419033, 0.276780, 0.396750, -0.0300848], # Week 10
            [0.524370, 0.519271, 0.542814, -0.0152694], # Week 11
            [0.465905, 0.784043, 0.602900, -0.0600895], # Week 12
        ]
    },

    "Function 4": {
        # Changed beta to 2.5 to help explore week 4
        # beta 0.01 found the peak
        #beta to 0.5 week 11
        #beta 0.1 week 12
        #beta to 0.01 week 13
        "beta": 0.01, 
        "nu": 2.5, 
        "data": [
            [0.896981054, 0.72562797, 0.175404309, 0.701694369, -22.1082878],
            [0.889356396, 0.499587855, 0.539268858, 0.508783439, -14.6013966],
            [0.250946243, 0.0336931305, 0.145380025, 0.494932421, -11.6999325],
            [0.346962061, 0.00625040024, 0.760563606, 0.613023557, -16.0537651],
            [0.124871181, 0.129770193, 0.384400483, 0.287076101, -10.0696334],
            [0.801302707, 0.500231094, 0.70664456, 0.195102841, -15.4870825],
            [0.247708262, 0.0604454273, 0.0421863451, 0.441324251, -12.681685],
            [0.746702242, 0.757091504, 0.36935306, 0.206566281, -16.0263998],
            [0.400665027, 0.0725742511, 0.886768254, 0.24384229, -17.0492346],
            [0.626070596, 0.586751259, 0.438805782, 0.778857694, -12.741766],
            [0.957135293, 0.597644383, 0.766113852, 0.776209905, -27.3163964],
            [0.732812426, 0.145249979, 0.476812718, 0.133365734, -13.5276489],
            [0.655115479, 0.0723918269, 0.687151746, 0.0815165642, -16.6791152],
            [0.219734429, 0.832031335, 0.482864162, 0.0825692306, -16.5071586],
            [0.48859419, 0.211965096, 0.939177907, 0.376191726, -17.8179993],
            [0.167130486, 0.876554558, 0.217239545, 0.959800985, -26.5618208],
            [0.216911188, 0.166085829, 0.241372256, 0.770062476, -12.7583242],
            [0.387487837, 0.804532258, 0.751795483, 0.723827439, -19.4415576],
            [0.985621893, 0.666932679, 0.156783283, 0.856534801, -28.9032737],
            [0.0378248285, 0.664853346, 0.161982175, 0.25392378, -13.7027469],
            [0.683486385, 0.902770103, 0.335419826, 0.999482561, -29.4270914],
            [0.170347305, 0.756959083, 0.276520486, 0.531231498, -11.565742],
            [0.859656919, 0.919592322, 0.206138728, 0.097796831, -26.8577864],
            [0.282138368, 0.505986912, 0.530530843, 0.096301623, -7.96677535],
            [0.326075785, 0.472366904, 0.453191996, 0.105887338, -6.70208925],
            [0.948389362, 0.894513008, 0.851637817, 0.552196286, -32.6256602],
            [0.66495539, 0.0465662767, 0.116777469, 0.79371778, -19.9894979],
            [0.577765614, 0.428771742, 0.425825867, 0.249007415, -4.02554228],
            [0.738613014, 0.482102634, 0.709366443, 0.503970014, -13.1227823],
            [0.854810797, 0.49396462, 0.735309975, 0.808092013, -23.1394284],
            # New weekly data points
            [0.419281, 0.437049, 0.353228, 0.437850, 0.3357274422869634],
            [0.442483, 0.466787, 0.477892, 0.455141, -2.01498],
            [0.440986, 0.495571, 0.241684, 0.407382, -2.75757], # Week 3
            [0.489649, 0.310165, 0.358270, 0.433197, -1.57828],
            [0.418799, 0.438960, 0.357347, 0.436386, 0.3440450], # Week 5
            [0.367723, 0.381390, 0.335715, 0.425916, 0.3108448745461705], # Week 6
            [0.381484, 0.387728, 0.322992, 0.454915, -0.9673647], # Week 7
            [0.411839, 0.426794, 0.395209, 0.342080, 0.4583850], # Week 8
            [0.403085, 0.418643, 0.361454, 0.391750, 0.5493858], # Week 9
            [0.264962, 0.352762, 0.376326, 0.354413, -1.152998], # Week 10
            [0.386665, 0.331691, 0.372851, 0.332545, -0.1731655], # Week 11
            [0.368686, 0.362030, 0.356817, 0.395233, 0.6088714], # Week 12
        ]
    },

    "Function 5": {
        "beta": 0.01,        # Lowered to 0.5 in week 3 # lowered to 0.01 in week 4
        "nu": 2.5,          # LOWER nu allows for "rougher" landscapes
        "data": [
            [0.191447084, 0.0381933714, 0.607417811, 0.414584137, 64.4434399],
            [0.758652949, 0.536517738, 0.656000382, 0.360341553, 18.3013796],
            [0.438349873, 0.804339705, 0.210245266, 0.151294816, 0.112939795],
            [0.706050834, 0.534191961, 0.264243345, 0.482087549, 4.21089813],
            [0.836477993, 0.193609647, 0.663892697, 0.785648883, 258.370525],
            [0.68343225, 0.118662642, 0.82904591, 0.567576606, 78.4343889],
            [0.55362148, 0.667349979, 0.323805819, 0.814869754, 57.5715369],
            [0.352356269, 0.322241532, 0.116979368, 0.473112522, 109.571876],
            [0.153785706, 0.72938169, 0.422598437, 0.443074166, 8.84799176],
            [0.463442267, 0.63002451, 0.107906456, 0.957643899, 233.22361],
            [0.677491148, 0.358509507, 0.479592224, 0.0728804811, 24.4230883],
            [0.583973412, 0.147242646, 0.348097462, 0.428614651, 64.4201468],
            [0.306888719, 0.316878127, 0.622634481, 0.0953990581, 63.4767158],
            [0.511141775, 0.817956997, 0.728710418, 0.112353623, 79.7291299],
            [0.438933376, 0.774091762, 0.378167086, 0.933696207, 355.806818],
            [0.224189023, 0.84648049, 0.87948418, 0.878515684, 1088.85962],
            [0.725261723, 0.479870486, 0.0889468426, 0.75976022, 28.8667516],
            [0.35548161, 0.639619367, 0.417617679, 0.12260384, 45.1815703],
            [0.119879226, 0.862540306, 0.643331326, 0.849803829, 431.612757],
            [0.12688467, 0.153429621, 0.770162188, 0.190518105, 9.97233189],
            # New weekly data points
            [0.269083, 0.909782, 0.983415, 0.958040, 2788.2743227306664],
            [0.805711, 0.486234, 0.289338, 0.791750, 121.9925],
            [0.301243, 0.903194, 0.970421, 0.953718, 2572.913], # Week 3
            [0.240335, 0.981963, 0.994803, 0.955230, 3602.395],
            [0.252715, 0.991518, 0.994538, 0.894159, 3127.222], # Week 5
            [0.197214, 0.984748, 0.989458, 0.974145, 3778.1452802596764], # Week 6
            [0.207535, 0.988725, 0.947408, 0.987915, 3510.191], # Week 7
            [0.024150, 0.990674, 0.996269, 0.986865, 4089.9015], # Week 8
            [0.184050, 0.969518, 0.981735, 0.968970, 3446.8235], # Week 9
            [0.018013, 0.981994, 0.980365, 0.988268, 3797.7959], # Week 10
            [0.133537, 0.966499, 0.975454, 0.984542, 3515.9145], # Week 11
            [0.415788, 0.070544, 0.050473, 0.362773, 128.2447], # Week 12
        ] 
    },

    "Function 6": {
        # increased to 5 in week 3
        # beta 1.96 week 6
        # beta 2.5 week 7
        # beta to 1 week 10
        #beta to 0.5 week 13
        "beta": 0.5,        
        "nu": 2.5,          # LOWER nu allows for "rougher" landscapes
        "data": [
            [0.7281861, 0.15469257, 0.73255167, 0.69399651, 0.05640131, -0.71426495],
            [0.24238435, 0.84409997, 0.5778091, 0.67902128, 0.50195289, -1.20995524],
            [0.72952261, 0.7481062, 0.67977464, 0.35655228, 0.67105368, -1.67219994],
            [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905, -1.53605771],
            [0.6188123, 0.33180214, 0.18728787, 0.75623847, 0.3288348, -0.82923655],
            [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115, -1.24704893],
            [0.14511079, 0.8966846, 0.89632223, 0.72627154, 0.23627199, -1.23378638],
            [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594, -1.69434344],
            [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624, -2.57116963],
            [0.75759436, 0.35583141, 0.0165229, 0.4342072, 0.11243304, -1.30911635],
            [0.5367969, 0.30878091, 0.41187929, 0.38822518, 0.5225283, -1.14478485],
            [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737, -1.91267714],
            [0.6293079, 0.80348368, 0.81140844, 0.04561319, 0.11062446, -1.62283895],
            [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173, -1.35668211],
            [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847, -2.0184254],
            [0.25890557, 0.79367771, 0.6421139, 0.19667346, 0.59310318, -1.70255784],
            [0.43216593, 0.71561781, 0.3418191, 0.70499988, 0.61496184, -1.29424696],
            [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599, -0.93575656],
            [0.9217762, 0.93187122, 0.41487637, 0.59505727, 0.73562569, -2.15576776],
            [0.12667892, 0.2914703, 0.06452848, 0.6805146, 0.89281919, -1.74688209],
            # New weekly data points
            [0.896676, 0.087385, 0.947572, 0.990532, 0.955831, -2.238834732813564],
            [0.668544, 0.807776, 0.106857, 0.959500, 0.823456, -2.15901],
            [0.420795, 0.694776, 0.340751, 0.677167, 0.570607, -1.17216], # Week 3
            [0.540039, 0.298335, 0.405373, 0.386144, 0.500009, -1.19073],
            [0.224989, 0.343347, 0.482874, 0.968819, 0.039910, -0.4959787], # Week 5
            [0.391818, 0.303949, 0.568747, 0.886431, 0.295158, -0.4933895168297094], # Week 6
            [0.340714, 0.031669, 0.865462, 0.947394, 0.021569, -0.8477095], # Week 7
            [0.245986, 0.419729, 0.020906, 0.946525, 0.025935, -1.108851], # Week 8
            [0.375853, 0.386905, 0.613153, 0.704287, 0.004579, -0.2558177], # Week 9
            [0.437031, 0.359954, 0.685246, 0.833493, 0.058688, -0.2424498], # Week 10
            [0.463947, 0.465143, 0.548083, 0.842884, 0.075093, -0.3622200], # Week 11
            [0.341536, 0.373270, 0.757125, 0.716376, 0.020222, -0.3580766], # Week 12
        ] 
    },
    # ... Repeat for 3, 4, 5, 6 ...

    "Function 7": {
        #beta 0.5 week 5
        #beta 0.1 week 6
        #beta 0.05 week 7
        #beta to 0.01 week 11
        #beta 0.05 week 12
        #beta to 0.01 week 13
        "beta": 0.01,        # HIGHER beta for high dimensions (More exploration)
        "nu": 1.5,          # LOWER nu allows for "rougher" landscapes
        "data": [
            [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362, 0.6044327],
            [0.54300258, 0.9246939, 0.34156746, 0.64648585, 0.71844033, 0.34313266, 0.56275307],
            [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654, 0.00750324],
            [0.11886697, 0.61505494, 0.90581639, 0.8553003, 0.41363143, 0.58523563, 0.0614243],
            [0.63021764, 0.8380969, 0.68001305, 0.73189509, 0.52673671, 0.34842921, 0.2730468],
            [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366, 0.08374657],
            [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984, 1.3649683],
            [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171, 0.09264495],
            [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164, 0.0178696],
            [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986, 0.03356494],
            [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637, 0.0735163],
            [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166, 0.2063097],
            [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079, 0.00882563],
            [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755, 0.26840032],
            [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776, 0.61152553],
            [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361, 0.01479818],
            [0.41762629, 0.06409998, 0.24566877, 0.5590408, 0.19153138, 0.25464092, 0.27489251],
            [0.72628566, 0.46489581, 0.92457051, 0.8072454, 0.6354384, 0.14341787, 0.06676325],
            [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825, 0.04211835],
            [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924, 0.00270147],
            [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429, 0.01820907],
            [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392, 0.00701603],
            [0.68685257, 0.04101721, 0.00757301, 0.285009, 0.69156848, 0.6555429, 0.10050661],
            [0.17597754, 0.6244165, 0.29554198, 0.46955276, 0.09776977, 0.72814108, 0.47539552],
            [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019, 0.67514163],
            [0.06661051, 0.52804507, 0.8160952, 0.96101714, 0.08650933, 0.77778822, 0.51645722],
            [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176, 0.00377748],
            [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983, 0.08130609, 0.00313433],
            [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764, 0.36288398, 0.02134252],
            [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547, 0.07966402, 0.09541116],
            # New weekly data points
            [0.027717, 0.410009, 0.349553, 0.169096, 0.332387, 0.685561, 1.9685506790729916],
            [0.138617, 0.055376, 0.694827, 0.264511, 0.051501, 0.533020, 0.88203],
            [0.048907, 0.333110, 0.456729, 0.028143, 0.300089, 0.744511, 1.53549], # Week 3
            [0.027644, 0.301105, 0.252762, 0.157405, 0.296209, 0.692165, 2.08653],
            [0.005699, 0.267150, 0.379692, 0.157250, 0.280376, 0.626919, 2.27409], # Week 5
            [0.016398, 0.176105, 0.408428, 0.210921, 0.222301, 0.684378, 2.4249953664991537], # Week 6
            [0.045057, 0.176089, 0.454466, 0.178831, 0.264456, 0.706152, 2.627131], # Week 7
            [0.006777, 0.139926, 0.454213, 0.243583, 0.428283, 0.737450, 2.214167], # Week 8
            [0.077269, 0.172148, 0.505603, 0.176296, 0.344754, 0.763852, 2.5944846], # Week 9
            [0.020434, 0.199570, 0.516766, 0.207244, 0.349715, 0.681735, 2.725886], # Week 10
            [0.039000, 0.157990, 0.579261, 0.203792, 0.248687, 0.719614, 2.564115], # Week 11
            [0.101159, 0.210334, 0.420198, 0.196378, 0.292318, 0.726913, 2.818316], # Week 12
        ] 
    },

    "Function 8": {
        #beta changed to 1.96 week 5
        #beta 1 week 6
        #beta 0.5 week 7
        #beta 0.01 week 8
        #beta to 0.05 week 9
        #beta to 0.01 week 10
        "beta": 0.01,        # Maximum exploration for the hardest function
        "nu": 1.5,
        "data": [
            [0.604994453, 0.29221502, 0.908452748, 0.355506242, 0.201668719, 0.575338005, 0.310310951, 0.734281377, 7.3987211],
            [0.178006959, 0.566222654, 0.994861845, 0.210325006, 0.320152657, 0.707908792, 0.635384489, 0.107131627, 7.00522736],
            [0.00907697668, 0.811626153, 0.52052036, 0.0756866752, 0.265111825, 0.0916516894, 0.592415145, 0.367320262, 8.45948162],
            [0.506028164, 0.653730123, 0.363410779, 0.177981049, 0.0937283044, 0.197425331, 0.7558269, 0.292472339, 8.28400781],
            [0.359909264, 0.249075679, 0.49599717, 0.709214981, 0.114987195, 0.289206921, 0.557295151, 0.593881726, 8.60611679],
            [0.778818344, 0.00341949948, 0.33798313, 0.519527778, 0.820906993, 0.537246689, 0.551347098, 0.660032086, 8.54174792],
            [0.908649322, 0.0622496998, 0.238259546, 0.766603545, 0.132335962, 0.990243814, 0.688067822, 0.742495941, 7.32743458],
            [0.586371444, 0.880735726, 0.745020752, 0.546034849, 0.00964887799, 0.748991763, 0.23090707, 0.0979156228, 7.29987205],
            [0.761137326, 0.85467239, 0.382124331, 0.337351983, 0.68970832, 0.309853052, 0.631379683, 0.0419560695, 7.95787474],
            [0.984933202, 0.699506258, 0.998885497, 0.180148456, 0.580143147, 0.231087191, 0.490826936, 0.31368272, 5.59219339],
            [0.112071314, 0.437735663, 0.596598785, 0.592775633, 0.22698177, 0.410104519, 0.921237577, 0.674752759, 7.85454099],
            [0.791887508, 0.576191336, 0.694528359, 0.283423782, 0.136755461, 0.279161861, 0.842767264, 0.625327922, 6.79198578],
            [0.143550296, 0.937414515, 0.232324818, 0.0090434854, 0.41457893, 0.409325169, 0.553778522, 0.2058408, 8.97655402],
            [0.769916548, 0.458759088, 0.559000445, 0.694604441, 0.503199022, 0.728346383, 0.784253534, 0.663131087, 7.3790829],
            [0.0564474111, 0.0659555534, 0.022928678, 0.0387864724, 0.403935441, 0.801055329, 0.488307007, 0.893084977, 9.598482],
            [0.862437445, 0.482733822, 0.281869398, 0.544102227, 0.88749026, 0.382654693, 0.601901993, 0.47646169, 8.15998319],
            [0.351511904, 0.590064942, 0.909436304, 0.678408354, 0.212825656, 0.0884603803, 0.410152995, 0.195724292, 7.13162397],
            [0.735903638, 0.0346118895, 0.728030269, 0.147426522, 0.295743139, 0.445117308, 0.975179686, 0.374339784, 6.76796253],
            [0.680293974, 0.255104646, 0.862187985, 0.134395821, 0.3263292, 0.287906871, 0.435010484, 0.364200126, 7.43374407],
            [0.0443292532, 0.0135814872, 0.25819824, 0.577644163, 0.051279923, 0.158563071, 0.591030124, 0.0779529335, 9.01307515],
            [0.77834548, 0.751145652, 0.314142208, 0.902985775, 0.335381656, 0.386322669, 0.748972486, 0.988755104, 7.31089382],
            [0.898887111, 0.523641705, 0.876783255, 0.218696449, 0.900260894, 0.282766245, 0.91107791, 0.472398218, 5.84106731],
            [0.145120286, 0.11932754, 0.420888224, 0.387608607, 0.155422833, 0.875171626, 0.510559672, 0.728610579, 9.14163949],
            [0.338954419, 0.566932018, 0.376751098, 0.098915729, 0.659451687, 0.24554809, 0.762482784, 0.732153467, 8.81755844],
            [0.176150018, 0.293961428, 0.975679966, 0.793936306, 0.923400762, 0.0308422938, 0.803254524, 0.595897583, 6.45194313],
            [0.0289466302, 0.0282790578, 0.481371555, 0.6131746, 0.672660448, 0.0221134069, 0.601483302, 0.524885053, 8.83074505],
            [0.192639868, 0.630677279, 0.416795837, 0.490529289, 0.796086023, 0.654567065, 0.276241193, 0.295517586, 9.34427428],
            [0.943185017, 0.218850618, 0.721184081, 0.424597072, 0.986902, 0.535182984, 0.714743177, 0.96009372, 6.88784639],
            [0.532721401, 0.833692597, 0.0713990037, 0.116811483, 0.73069311, 0.937375591, 0.866507981, 0.127901999, 8.04221254],
            [0.447095841, 0.843952527, 0.729546115, 0.639151378, 0.409287137, 0.132645694, 0.0359088762, 0.44683847, 7.69236805],
            [0.382224965, 0.557135837, 0.853101634, 0.333795692, 0.265721272, 0.480872916, 0.237647062, 0.768631956, 7.92375877],
            [0.53281953, 0.862308484, 0.538267119, 0.0494429349, 0.719701189, 0.906705899, 0.108230943, 0.525347913, 8.42175924],
            [0.394865187, 0.331801666, 0.740754301, 0.697861725, 0.737404439, 0.78377681, 0.254495461, 0.87114551, 8.2780624],
            [0.98594539, 0.873053629, 0.0703926194, 0.0535872927, 0.734152958, 0.520258522, 0.811040045, 0.103360365, 7.11345716],
            [0.964573386, 0.973979787, 0.663753351, 0.662215992, 0.673121672, 0.905237624, 0.458874624, 0.560917502, 6.40258841],
            [0.472070709, 0.168202645, 0.0864275662, 0.452655513, 0.48061922, 0.622439489, 0.928974462, 0.112536267, 8.47293632],
            [0.856006953, 0.638893704, 0.326192022, 0.668503115, 0.240298369, 0.21029889, 0.167546362, 0.963589863, 7.97768459],
            [0.810031736, 0.635046041, 0.269547579, 0.869605338, 0.66192159, 0.252258727, 0.765670033, 0.890548667, 7.46087219],
            [0.796252524, 0.00703653252, 0.35569738, 0.487566053, 0.740519615, 0.706650103, 0.992914495, 0.381734367, 7.43659353],
            [0.481245331, 0.102460721, 0.219485939, 0.677322369, 0.247509187, 0.244340858, 0.163824527, 0.71596164, 9.18300525],
            # New weekly data points
            [0.094400, 0.211765, 0.000829, 0.047273, 0.695000, 0.390891, 0.029115, 0.502083, 9.8587346723031],
            [0.037836, 0.108393, 0.033738, 0.669339, 0.805440, 0.782939, 0.066865, 0.601450, 9.57750],
            [0.082173, 0.583614, 0.014819, 0.097106, 0.266999, 0.175719, 0.064105, 0.998140, 9.46875], # Week 3
            [0.039746, 0.356781, 0.008482, 0.592545, 0.772677, 0.169634, 0.095650, 0.175247, 9.56050],
            [0.062784, 0.401458, 0.023356, 0.333327, 0.459519, 0.775625, 0.116858, 0.566441, 9.71840], # Week 5
            [0.008290, 0.156810, 0.068239, 0.152764, 0.614543, 0.409750, 0.279153, 0.793716, 9.9300560770329], # Week 6
            [0.084027, 0.145103, 0.073378, 0.363783, 0.723002, 0.374561, 0.250098, 0.493813, 9.9192979463641], # Week 7
            [0.105166, 0.132733, 0.007265, 0.170055, 0.501491, 0.401820, 0.251306, 0.688447, 9.89381], # Week 8
            [0.053742, 0.245050, 0.190346, 0.200382, 0.660498, 0.370313, 0.236614, 0.619573, 9.9439540], # Week 9
            [0.075689, 0.344862, 0.120634, 0.239579, 0.751891, 0.538500, 0.298840, 0.606540, 9.930376], # Week 10
            [0.007962, 0.317187, 0.120356, 0.268175, 0.691665, 0.588223, 0.122412, 0.557151, 9.914987], # Week 11
            [0.102471, 0.090030, 0.184132, 0.199700, 0.495122, 0.460653, 0.320674, 0.617552, 9.907951], # Week 12
        ] 
    }
}

In [3]:
# ==============================================================================
# 2. THE ADAPTIVE ENGINE
# ==============================================================================
def get_next_query(config, func_name):
    raw_data = config.get("data", [])
    if not raw_data or len(raw_data) < 1:
        return "No data"
        
    data = np.array(raw_data)
    X_train = data[:, :-1]
    y_train = data[:, -1]
    dim = X_train.shape[1]
    
    # 1. DYNAMIC KERNEL
    # We pull the 'nu' value from your config above
    nu_val = config.get("nu", 2.5)
    kernel = Matern(nu=nu_val, length_scale_bounds=(1e-10, 1e5))
    
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True, alpha=1e-5)
    gp.fit(X_train, y_train)

    # 2. DYNAMIC CANDIDATES
    # 8D needs way more random guesses than 2D to find a good spot
    if dim <= 2: n_candidates = 20000
    elif dim <= 5: n_candidates = 50000
    else: n_candidates = 100000  # For Function 7 & 8
    
    X_candidates = np.random.uniform(0, 1, (n_candidates, dim))

    # 3. PREDICT & OPTIMIZE
    mean, std = gp.predict(X_candidates, return_std=True)
    
    # We pull the 'beta' value from your config above
    beta = config.get("beta", 1.96)
    acquisition_values = mean + (beta * std)
    
    best_index = np.argmax(acquisition_values)
    return np.array2string(X_candidates[best_index], separator='-', precision=6).replace('[', '').replace(']', '')

In [4]:
# ==============================================================================
# 3. EXECUTION
# ==============================================================================
print(f"{'FUNCTION':<12} | {'DIM':<5} | {'BETA':<5} | {'NEXT QUERY'}")
print("-" * 60)

for name, config in database.items():
    data = config.get("data", [])
    if not data:
        print(f"{name:<12} | {'-':<5} | {config.get('beta'):<5} | SKIP (No Data)")
        continue
        
    dim = len(data[0]) - 1
    result = get_next_query(config, name)
    print(f"{name:<12} | {dim:<5} | {config.get('beta'):<5} | {result}")

FUNCTION     | DIM   | BETA  | NEXT QUERY
------------------------------------------------------------
Function 1   | 2     | 5     | 0.65543-0.99524
Function 2   | 2     | 0.1   | 0.478886-0.512468
Function 3   | 3     | 5     | 0.764628-0.632567-0.657627
Function 4   | 4     | 0.01  | 0.454176-0.453983-0.384989-0.407865
Function 5   | 4     | 0.01  | 0.150238-0.946325-0.989899-0.999488
Function 6   | 5     | 0.5   | 0.42881 -0.348968-0.524555-0.823247-0.021222
Function 7   | 6     | 0.01  | 0.132084-0.142546-0.495076-0.272626-0.337368-0.680737
Function 8   | 8     | 0.01  | 0.082783-0.251705-0.16135 -0.26308 -0.508964-0.380318-0.218296-0.57496 
